# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [31]:
# imports
import os
import time 
from dotenv import load_dotenv
# from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

In [32]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

In [33]:
# set up environment
load_dotenv(override=True)

open_ai_key = os.getenv('OPENAI_API_KEY')
# check to make sure the correct openai key is used 
key_starts_with ='sk-proj'
key_ends_with ='lFEA'

if not open_ai_key:
    print(f"OpenAI key not found.\n Please make sure to add OPEN_AI_KEY to the .env file.\n")
elif not open_ai_key.startswith(key_starts_with) and open_ai_key.endswith(key_ends_with):
    print(f"OpenAI key found but doesn't starts with {key_starts_with}, and doesn't end with {key_ends_with}.")
else:
    print("OPEN_AI_KEY looks good, let's start!\n")


OPEN_AI_KEY looks good, let's start!



In [34]:
# here is the question; type over this to ask something new

question = """
Please explain why python module uv is better than pip. Also give a few examples for comparison. 
"""

In [35]:
# Get gpt-4o-mini to answer, with streaming

openai = OpenAI()

# start time 
start_time = time.time()
response = openai.chat.completions.create(model="gpt-6-luna", reasoning_effort="none", messages=[{"role": "user", "content": question}])

result = response.choices[0].message.content
end_time = time.time()
display(Markdown(result))
print(f"Time taken for gpt-6-luna to answer: {end_time - start_time:.2f} seconds")

`uv` can be a better choice than `pip` for many everyday Python workflows, especially when you also want to manage Python versions, virtual environments, and project dependencies. It is not a drop-in replacement for every pip workflow, though, and `pip` remains widely supported and perfectly suitable for installing packages.

### Main differences

- **Speed:** `uv` is implemented in Rust and is often much faster at resolving and installing dependencies, particularly in larger environments.
- **Broader scope:** `pip` primarily installs packages. `uv` also provides commands for creating environments, managing Python versions, and maintaining project dependencies and lockfiles.
- **Reproducible projects:** `uv` can use a `pyproject.toml` and a lockfile (`uv.lock`) to keep dependency choices consistent across machines.
- **Familiarity and compatibility:** `pip` is the standard installer and is supported by nearly every Python guide and tool. `uv` offers a pip-compatible interface, but some specialized workflows may still require `pip` or another tool.

### Examples

**Install a package**

```bash
# pip
python -m pip install requests

# uv
uv pip install requests
```

The `uv` command aims to be a faster alternative here. In this form, it usually installs into the active environment; create or activate a virtual environment first if you want an isolated install.

**Create a virtual environment**

```bash
# Traditional approach
python -m venv .venv
source .venv/bin/activate
python -m pip install requests

# With uv
uv venv
uv pip install requests
```

`uv` can also create an environment using a selected Python version, provided that version is available or can be installed:

```bash
uv venv --python 3.12
```

**Manage project dependencies**

With `pip`, a common workflow is to install packages and record them in a requirements file:

```bash
python -m pip install requests
python -m pip freeze > requirements.txt
```

With `uv`, you can declare dependencies in a project and synchronize the environment from the project metadata and lockfile:

```bash
uv init
uv add requests
uv sync
```

`uv add` updates the project’s dependency information; `uv sync` installs the versions selected for the project.

### Which should you use?

Choose **uv** if you want a fast, unified tool for project dependencies, virtual environments, and Python versions. Choose **pip** if you want the standard, familiar installer or need to follow tooling that specifically expects pip. You can also use both: `uv` provides pip-compatible commands, while standard pip remains available in the Python ecosystem.

Time taken for gpt-6-luna to answer: 6.27 seconds


In [36]:
# Get Llama 3.2 to answer
load_dotenv(override=True)

base_url = os.getenv("OLLAMA_BASE_URL")
ollama = OpenAI(base_url=base_url, api_key='ollama')

# check if ollama is running 
import subprocess 

try:
    result = subprocess.run(
        ["ollama", "list"],
        capture_output=True, 
        text=True, 
        timeout=5
    )
    if result.returncode == 0:
        print("Ollama is running!")
        print(result.stdout)
    else:
        print("Ollama command faild:")
        print(result.stderr)
except FileNotFoundError:
    print("Ollama is not installed or not in PATH")
except subprocess.TimeoutExpired:
    print("Ollama command timed out.")

Ollama is running!
NAME                  ID              SIZE     MODIFIED    
qwen3:30b-thinking    ad815644918f    18 GB    3 weeks ago    



In [37]:
# ollama time 
start_time = time.time()

response = ollama.chat.completions.create(model="qwen3:30b-thinking", messages=[{"role": "user", "content": question}])
results  = response.choices[0].message.content

end_time = time.time()

print(display(Markdown(results)))
print(f"Time taken for Llama 3.2 to answer: {end_time - start_time:.2f} seconds")

You've hit on a **common misconception**: **`uv` is *not* a Python module**—it's a **standalone, high-performance Python package installer and resolver**, built in Rust. **`pip` is the standard Python package installer** (a Python module itself). They are *different tools*, not competitors within the same ecosystem. 

Let's clarify and then compare them properly:

---

### 🔧 **Key Clarifications**
| Tool         | Type                | Language | Purpose                                  |
|--------------|---------------------|----------|------------------------------------------|
| `uv`         | **Standalone CLI Tool** | Rust     | Fast package installation/resolution       |
| `pip`        | **Python Module**   | Python   | Standard package installer (part of Python) |

**Why the confusion?**  
- `uv` is designed to *replace* `pip` in performance-critical scenarios (it’s **not** a module you install via `pip install uv`).
- It’s **not meant to be used as a Python module**—you run it as `uv pip install ...` (similar to `pip install ...`).

---

### ⚡ **Why `uv` is (often) "Better" Than `pip`**  
`uv` is **faster** and **more efficient** in most cases due to its Rust implementation and optimized algorithms. Here’s why:

1. **Rust Performance**  
   - `uv` uses Rust (zero-cost abstractions, parallelism, no GIL), while `pip` is Python (slow due to GIL, interpreted overhead).
2. **Optimized Dependency Resolution**  
   - `uv` uses a **much faster resolver** (based on the `resolver` crate), resolving dependencies in **milliseconds vs. seconds**.
3. **Parallel Downloads**  
   - `uv` downloads packages concurrently (unlike `pip`, which is sequential by default).
4. **Efficient Wheel Building**  
   - Builds wheels faster via Rust’s `build` API (no Python overhead).
5. **Simpler Codebase**  
   - `uv` is a single binary (no Python interpreter needed for resolution).

> 💡 **Real-World Impact**: `uv` can install packages **2–10x faster** than `pip` (especially with large dependency trees).

---

### 📊 **Comparative Examples**
Let’s compare installing `requests` (a common package) as an example.

#### Example 1: Simple Install Speed
| Command                           | Time (on my machine) | Notes                                      |
|-----------------------------------|----------------------|--------------------------------------------|
| `pip install requests`            | 1.2 seconds          | Standard `pip` (uses Python resolver)      |
| `uv pip install requests`         | **0.3 seconds**      | `uv` uses its own resolver/download engine |

> 💡 **Why?** `uv` downloads the package *in parallel* and skips redundant resolver steps.

#### Example 2: Complex Dependency Resolution
Install `pandas` (which depends on 10+ packages):
```bash
# Using pip
pip install pandas  # Resolves dependencies one-by-one, slow + may hit network timeout

# Using uv
uv pip install pandas  # Resolves all dependencies in <1 second
```
**`uv`’s advantage**: Its resolver (based on [Pip’s resolver](https://github.com/pypa/pip/issues/988#issuecomment-1426148704)) is **10x faster** for complex cases.  
*(Source: [uv’s benchmarks](https://github.com/astral-sh/uv#performance))*

#### Example 3: Building Wheels (from Source)
```bash
# pip (slow, uses Python-based builder)
pip install numpy  # Builds wheel from source in ~12 seconds

# uv (fast, uses Rust-based build)
uv pip install numpy  # Builds wheel in ~2 seconds
```
> 💡 **Why?** `uv` uses Rust’s `build` system directly (avoids Python’s `setuptools` overhead).

---

### ⚠️ **Important Caveats**
1. **`uv` is NOT a drop-in replacement for `pip`**:
   - Missing features: `uv pip` doesn’t support `--user`, `--no-cache-dir`, or some rare `pip` flags.
   - Less mature: Still in active development (v0.2.0 as of 2024).
2. **You don’t install `uv` via `pip`**:
   - Download it directly: [https://github.com/astral-sh/uv](https://github.com/astral-sh/uv)
   - **Do not run `pip install uv`** (this would try to install a non-existent Python module named `uv`).
3. **Not all projects work perfectly with `uv`**:
   - Some edge cases (e.g., custom build scripts) may fail. Test in your workflow first.

---

### ✅ **When to Use `uv`**
| Scenario                          | Recommendation |
|-----------------------------------|----------------|
| Installing packages in CI/CD       | ✅ **Use `uv`** (speeds up builds) |
| Development on a local machine     | ✅ **Use `uv`** (if you need speed) |
| Legacy projects requiring `pip` flags | ⚠️ **Stick to `pip`** (e.g., `--no-cache-dir`) |
| Debugging package conflicts        | ⚠️ **Use `pip`** (more debugging output) |

---

### 🧪 **Try It Yourself (Test on Your Machine)**
1. Install `uv` (as a standalone binary):
   ```bash
   # macOS (Homebrew)
   brew install uv

   # Linux (curl)
   curl -LsSf https://astral.sh/uv/install.sh | sh

   # Windows (PowerShell)
   Invoke-WebRequest -Uri https://astral.sh/uv/install.ps1 -OutFile install.ps1
   .\install.ps1
   ```
2. Benchmark:
   ```bash
   time uv pip install requests
   time pip install requests
   ```

---

### 📌 **Summary**
| Feature                | `pip`                          | `uv` (the tool)               |
|------------------------|--------------------------------|-------------------------------|
| **Language**           | Python (slow, GIL-bound)       | **Rust (fast, parallel)**     |
| **Speed**              | Slow (seconds)                 | **Fast (milliseconds)**       |
| **Dependency Resolution** | Complex, slow                | **Optimized, 10x faster**     |
| **Installation**       | Use `pip` (standard)           | Use `uv` (download & run)     |
| **Maturity**           | Mature (used everywhere)       | Early-stage (v0.2.0)          |

> 💡 **Bottom line**: `uv` is **not a Python module**, but a **Rust-based tool** built to *replace `pip` for speed*. Use it for **fast installations**—but **don’t replace `pip` in all scenarios** until it matures. For most users, `uv` is the future of Python package installation.

For more: [https://github.com/astral-sh/uv](https://github.com/astral-sh/uv) (the official repo).

None
Time taken for Llama 3.2 to answer: 35.80 seconds


In [38]:
# Deepseek API 
load_dotenv(override=True)

deepseek_url = "https://api.deepseek.com"
deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")

# check to make sure the correct deepseek key is used
if not deepseek_api_key:
    print(f"Deepseek API key not found.\n Please make sure to add DEEPSEEK_API_KEY to the .env file.\n")
elif not deepseek_api_key.startswith("sk-"):
    print(f"Deepseek API key found but doesn't start with 'sk-'.")
elif not deepseek_api_key.endswith("3a0f"):
    print(f"Deepseek API key found but doesn't end with '3a0f'.")
else:
    print("DEEPSEEK_API_KEY looks good, let's start!\n")



DEEPSEEK_API_KEY looks good, let's start!



In [39]:
deepseek = OpenAI(base_url=deepseek_url, api_key=deepseek_api_key)

# deepseek time 
start_time = time.time()
response = deepseek.chat.completions.create(model="deepseek-v4-pro", messages=[{"role": "user", "content": question}])

result = response.choices[0].message.content
if result:
    display(Markdown(result))
end_time = time.time()
print(f"Time taken for deepseek-v4-pro to answer: {end_time - start_time:.2f} seconds")
    

`uv` is not technically a Python module like `pip`, but a standalone command-line tool written in Rust. It can be installed via `pip install uv` and invoked as `uv` or `python -m uv`. It is designed to be a **drop‑in replacement for `pip`**, but it is much faster and includes many additional features that go beyond basic package installation.

Here’s why `uv` is often considered better than `pip`:

## 1. Speed
`uv` is dramatically faster than `pip` because:
- It downloads packages **in parallel**.
- It uses a **global content‑addressed cache** (packages are stored once and hardlinked into virtual environments).
- Its core is written in Rust, which is highly optimised for I/O and CPU‑bound tasks.

For typical installs, `uv` can be **10–100× faster** than `pip`.

## 2. All‑in‑one tool
`uv` replaces not only `pip`, but also several other tools:
- `pip-tools` → `uv pip compile`
- `virtualenv` / `venv` → `uv venv`
- `pipx` → `uv tool`
- `pyenv` → `uv python`
- `poetry` / `pipenv` → `uv add`, `uv sync`, `uv lock`

This means you can manage Python versions, create virtual environments, compile requirements, and install project dependencies with a single tool.

## 3. Advanced dependency resolution
`uv` uses a robust, deterministic resolver that avoids many of the dependency conflicts `pip` can run into. It also produces a lockfile (`uv.lock`) that ensures everyone working on a project gets exactly the same versions.

## 4. Intelligent caching
`uv` maintains a global cache of downloaded and extracted packages. Once a package is cached, installing it again is almost instantaneous and does not consume extra disk space because the same files are hardlinked into new environments.

## 5. Modern project workflow
For projects, `uv` can manage `pyproject.toml` dependencies directly:
- `uv add requests` – adds `requests` to `pyproject.toml`, updates the lockfile, and installs it.
- `uv remove requests` – removes it.
- `uv sync` – installs all dependencies from the lockfile.
- `uv run` – runs a command in an ephemeral or project environment.

---

## Example comparisons

### 1. Installing a package
**pip**
```bash
pip install requests numpy pandas
```

**uv**
```bash
uv pip install requests numpy pandas
```

The `uv` command resolves and downloads packages in parallel, often finishing in a fraction of the time. On a second install, it uses the cache and completes almost instantly.

---

### 2. Creating a virtual environment and installing from `requirements.txt`
**pip**
```bash
python -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
```

**uv**
```bash
uv venv
source .venv/bin/activate
uv pip install -r requirements.txt
```

`uv venv` is much faster and can even download a Python interpreter for you if the requested version is not installed:
```bash
uv venv --python 3.12
```

---

### 3. Compiling reproducible requirements
**pip-tools (traditional)**
```bash
pip-compile requirements.in -o requirements.txt
```

**uv**
```bash
uv pip compile requirements.in -o requirements.txt
```

`uv` produces the same output format but does it much faster.

---

### 4. Project dependency management
**pip + manual workflow**
```bash
# Edit pyproject.toml or setup.py
pip install -e .
```

**uv**
```bash
uv add requests
uv sync
```

`uv add` automatically updates `pyproject.toml` and the lockfile, then installs the dependencies.

---

### 5. Running a script with dependencies
**pip**
```bash
pip install rich
python script.py
```

**uv**
```bash
uv run --with rich script.py
```

`uv run` creates a temporary environment, installs `rich`, runs the script, and cleans up afterwards—no permanent installation needed.

Alternatively, if the script contains inline metadata (PEP 723), you can simply use:
```bash
uv run script.py
```

---

## Caveats
- `uv` is **not part of the Python standard library**; it must be installed separately (though it can be installed via `pip`).
- Some obscure or rarely used `pip` options may not be supported.
- For very simple use cases, `pip` remains perfectly adequate and is still the default for most Python installations.

In short, `uv` is better than `pip` when you value speed, reproducibility, and a unified workflow. It is especially useful for developers who frequently create virtual environments, work with multiple Python versions, or need lockfile‑based dependency management.

Time taken for deepseek-v4-pro to answer: 33.91 seconds
